# PHAN TICH CAM XUC REVIEW THUONG MAI DIEN TU (SENTIMENT ANALYSIS)

Project nay thuc hien:
1. Tai du lieu review tu Hugging Face.
2. Tien xu ly tieng Viet (chuyen chu thuong, xoa ky tu dac biet, chuan hoa teencode, tach tu).
3. Phan tich du lieu EDA.
4. Trinh bay mo hinh Baseline (TF-IDF + Naive Bayes va Logistic Regression).
5. Trinh bay mo hinh Advanced (PyTorch Bi-LSTM).
6. So sanh, danh gia va chay thu review truc tiep.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import time
import os

# Bat progress bar cua tqdm cho pandas
tqdm.pandas()

# Import tien xu ly tu utils
from utils import preprocess_review
print("Import thanh cong va san sang!")


## 1. Doc Du Lieu Tu File Local


In [ ]:
# Doc du lieu tu file jsonl local da tai ve
df_raw = pd.read_json("shopee_reviews_dataset.jsonl", lines=True)
print("Dinh dang du lieu goc:")
print(df_raw.head())


## 2. Chuan Hoa Nhan Va Chia Tap Du Lieu


In [ ]:
from sklearn.model_selection import train_test_split

# Map label: positive -> 1, negative -> 0
df_raw['sentiment'] = df_raw['label'].map({'positive': 1, 'negative': 0})
df = df_raw[['review', 'sentiment']].rename(columns={'review': 'comment'}).copy()

# Chia tap du lieu: 80% Train / 10% Val / 10% Test
df_train_full, df_test = train_test_split(df, test_size=0.1, random_state=42, stratify=df['sentiment'])
df_train, df_val = train_test_split(df_train_full, test_size=0.1111, random_state=42, stratify=df_train_full['sentiment'])

print("So luong mau tap Train:", len(df_train))
print("So luong mau tap Val:", len(df_val))
print("So luong mau tap Test:", len(df_test))
print("\nPhan phoi nhan trong tap Train:")
print(df_train['sentiment'].value_counts())


## 3. Chay Tien Xu Ly Van Ban (Clean & Tokenize)


In [ ]:
# Chay tien xu ly du lieu (bao gom xoa ky tu, thay teencode, va tach tu)
print("Dang tien xu ly tap train...")
df_train['clean_comment'] = df_train['comment'].progress_apply(preprocess_review)
print("Dang tien xu ly tap val...")
df_val['clean_comment'] = df_val['comment'].progress_apply(preprocess_review)
print("Dang tien xu ly tap test...")
df_test['clean_comment'] = df_test['comment'].progress_apply(preprocess_review)

# Loai bo cac dong rong sau khi lam sach
df_train = df_train[df_train['clean_comment'] != ""].dropna()
df_val = df_val[df_val['clean_comment'] != ""].dropna()
df_test = df_test[df_test['clean_comment'] != ""].dropna()

print("\nVi du truoc va sau khi tien xu ly:")
print(df_train[['comment', 'clean_comment', 'sentiment']].head(3))


## 4. Phan Tich Du Lieu EDA


In [ ]:
# Ve bieu do phan phoi nhan va chieu dai cau
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(x='sentiment', data=df_train, palette='Set2')
plt.title("Phan phoi nhan (0: Negative, 1: Positive)")
plt.xlabel("Nhan")
plt.ylabel("So luong")

plt.subplot(1, 2, 2)
word_counts = df_train['clean_comment'].apply(lambda x: len(x.split()))
sns.histplot(word_counts, bins=40, color='teal', kde=True)
plt.title("Phan phoi so luong tu trong review")
plt.xlabel("So luong tu")
plt.ylabel("Tan suat")

plt.tight_layout()
plt.show()


## 5. Mo Hinh Baseline (Machine Learning Truyen Thong)
Su dung TF-IDF (N-gram) de vector hoa review, sau do chay Naive Bayes va Logistic Regression.


In [ ]:
# Vector hoa bang TF-IDF (ngram tu 1 den 2 tu, giu 10,000 tu quan trong)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_tfidf = vectorizer.fit_transform(df_train['clean_comment'])
X_test_tfidf = vectorizer.transform(df_test['clean_comment'])

y_train = df_train['sentiment'].values
y_test = df_test['sentiment'].values

# 1. Naive Bayes
t0 = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_time = time.time() - t0

y_pred_nb = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb, average='macro')

# 2. Logistic Regression
t0 = time.time()
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_time = time.time() - t0

y_pred_lr = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr, average='macro')

print(f"Naive Bayes -> Acc: {nb_acc*100:.2f}%, F1: {nb_f1*100:.2f}%, Time: {nb_time:.4f}s")
print(f"Logistic Regression -> Acc: {lr_acc*100:.2f}%, F1: {lr_f1*100:.2f}%, Time: {lr_time:.4f}s")


In [ ]:
# Ve confusion matrix cho ca 2 mo hinh baseline
cm_nb = confusion_matrix(y_test, y_pred_nb)
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
ax[0].set_title("Naive Bayes Confusion Matrix")
ax[0].set_xlabel("Du doan")
ax[0].set_ylabel("Thuc te")

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', ax=ax[1], xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
ax[1].set_title("Logistic Regression Confusion Matrix")
ax[1].set_xlabel("Du doan")
ax[1].set_ylabel("Thuc te")
plt.show()


## 6. Mo Hinh Advanced (Deep Learning - PyTorch Bi-LSTM)
Huan luyen mang no-ron tuan tu hai chieu (Bi-LSTM) tren moi truong PyTorch de nam bat thong tin ngu canh tieng Viet tot hon.


In [ ]:
from collections import Counter

# Buoc 1: Xay dung tu dien (vocab) tu tap train
all_words = []
for text in df_train['clean_comment']:
    all_words.extend(text.split())

vocab_counter = Counter(all_words)
# Loc cac tu co tan suat >= 2 truoc khi danh so index
filtered_words = [word for word, count in vocab_counter.items() if count >= 2]
vocab = {word: idx + 2 for idx, word in enumerate(filtered_words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

vocab_size = len(vocab)
print("Kich thuoc Vocab:", vocab_size)

# Ham bien doi tu thanh chi so (indices)
def text_to_sequence(text, vocab):
    return [vocab.get(word, vocab['<UNK>']) for word in text.split()]

# Tham so padding
max_len = 80

# PyTorch Dataset
class ReviewDataset(Dataset):
    def __init__(self, df, vocab, max_len):
        self.labels = df['sentiment'].values
        self.sequences = [text_to_sequence(text, vocab) for text in df['clean_comment']]
        self.max_len = max_len
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # Pad hoac cat
        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

train_dataset = ReviewDataset(df_train, vocab, max_len)
val_dataset = ReviewDataset(df_val, vocab, max_len)
test_dataset = ReviewDataset(df_test, vocab, max_len)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
# Dinh nghia kien truc mang Bi-LSTM
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, 
                           hidden_dim, 
                           num_layers=n_layers, 
                           bidirectional=True, 
                           batch_first=True,
                           dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        
        # Mean pooling de lay trung binh cac buoc thoi gian cua lstm output
        pooled = torch.mean(output, dim=1)
        return self.fc(self.dropout(pooled))


In [ ]:
# Khoi tao model, device va thiet lap loss/optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Model dang chay tren thiet bi:", device)

model = BiLSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=100, # Embedding 100 chieu
    hidden_dim=128,    # Hidden size 128
    output_dim=1,      # Output logic phan loai nhi phan
    n_layers=2,        # 2 lop LSTM xep chong
    dropout=0.5
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'So luong tham so co the train cua model Bi-LSTM: {count_parameters(model):,}')


In [ ]:
# Vong lap huan luyen model PyTorch
epochs = 8
best_val_loss = float('inf')
train_losses, val_losses = [], []
train_accs, val_accs = [], []

t0_lstm = time.time()
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(seqs).squeeze(1)
        loss = criterion(predictions, labels)
        
        preds = torch.round(torch.sigmoid(predictions))
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    train_loss = epoch_loss / len(train_loader)
    train_acc = correct / total
    
    # Validation loop
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            predictions = model(seqs).squeeze(1)
            loss = criterion(predictions, labels)
            
            preds = torch.round(torch.sigmoid(predictions))
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            val_loss += loss.item()
            
    valid_loss = val_loss / len(val_loader)
    valid_acc = val_correct / val_total
    
    train_losses.append(train_loss)
    val_losses.append(valid_loss)
    train_accs.append(train_acc)
    val_accs.append(valid_acc)
    
    print(f'Epoch {epoch+1:02} | Train Loss: {train_loss:.3f} Acc: {train_acc*100:.2f}% | Val Loss: {valid_loss:.3f} Acc: {valid_acc*100:.2f}%')
    
    # Luu lai trong so tot nhat
    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        torch.save(model.state_dict(), 'best_lstm.pt')
        
lstm_time = time.time() - t0_lstm
print(f"Hoan thanh huan luyen Bi-LSTM trong {lstm_time:.2f} giay!")


In [ ]:
# Ve do thi loss va accuracy cua Bi-LSTM
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(val_losses, label='Val Loss', color='orange')
plt.title('Loss qua cac epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc', color='blue')
plt.plot(val_accs, label='Val Acc', color='orange')
plt.title('Accuracy qua cac epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Tinh toan metrics cua Bi-LSTM tren test set
model.load_state_dict(torch.load('best_lstm.pt'))
model.eval()

test_preds = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        predictions = model(seqs).squeeze(1)
        preds = torch.round(torch.sigmoid(predictions))
        test_preds.extend(preds.cpu().numpy())
        
test_preds = np.array(test_preds)
lstm_acc = accuracy_score(y_test, test_preds)
lstm_f1 = f1_score(y_test, test_preds, average='macro')

print(f"Bi-LSTM Accuracy: {lstm_acc*100:.2f}%")
print(f"Bi-LSTM F1-Score: {lstm_f1*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=['Negative', 'Positive']))


In [ ]:
# Ve confusion matrix cho Bi-LSTM
cm_lstm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title("Confusion Matrix - Bi-LSTM")
plt.xlabel("Du doan")
plt.ylabel("Thuc te")
plt.show()


## 7. So Sanh Tong Hop Va Danh Gia
Bang so sanh giua cac phuong phap da su dung.


In [ ]:
summary_data = {
    'Mo hinh': ['Naive Bayes (Baseline)', 'Logistic Regression (Baseline)', 'PyTorch Bi-LSTM (Advanced)'],
    'Accuracy (%)': [nb_acc*100, lr_acc*100, lstm_acc*100],
    'F1-Score (%)': [nb_f1*100, lr_f1*100, lstm_f1*100],
    'Thoi gian train (s)': [nb_time, lr_time, lstm_time]
}
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))


## 8. Du Doan Review Truc Tiep (Interactive Tester)
Nhap review tieng Viet bat ky de mo hinh du doan cam xuc cua review do.


In [ ]:
def predict_sentiment(review_text):
    # Buoc 1: Tien xu ly tuong tu luc train
    cleaned = preprocess_review(review_text)
    print(f"Review goc: {review_text}")
    print(f"Da tien xu ly: {cleaned}")
    
    # Buoc 2: Du doan tu Logistic Regression (TF-IDF)
    tfidf_vec = vectorizer.transform([cleaned])
    lr_pred = lr_model.predict(tfidf_vec)[0]
    lr_prob = lr_model.predict_proba(tfidf_vec)[0]
    lr_label = "Positive" if lr_pred == 1 else "Negative"
    lr_confidence = lr_prob[1] if lr_pred == 1 else lr_prob[0]
    
    # Buoc 3: Du doan tu PyTorch Bi-LSTM
    seq = text_to_sequence(cleaned, vocab)
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
        
    model.eval()
    with torch.no_grad():
        tensor_seq = torch.tensor([seq], dtype=torch.long).to(device)
        logit = model(tensor_seq).squeeze(1).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
        lstm_label = "Positive" if prob >= 0.5 else "Negative"
        lstm_confidence = prob if prob >= 0.5 else (1.0 - prob)
        
    print("-" * 60)
    print(f"Logistic Regression: {lr_label:<10} (Do tin cay: {lr_confidence*100:.2f}%)")
    print(f"PyTorch Bi-LSTM:     {lstm_label:<10} (Do tin cay: {lstm_confidence*100:.2f}%)")
    print("-" * 60)
    print()

# Review test 1: Review rat tot
predict_sentiment("San pham tuyet voi giao hang rat nhanh va dong goi ky cang")

# Review test 2: Review phu dinh
predict_sentiment("Dung chan lam moi nguoi oi dung mua nha phi tien thuc su")

# Review test 3: Cau phu dinh phuc tap (Bi-LSTM se bat tot hon)
predict_sentiment("Sieu pham day nhung ma chat luong khong he tot")

